# FineSight-Perception Dataset Generation

**Dataset One: Fine-Grained Visual Perception Evaluation Set**

Goal: evaluate whether a VLM can identify extremely small-scale targets or subtle visual changes.

| Task | Description | Sizes | Images |
|------|-------------|-------|--------|
| Letter recognition | Identify a single rendered letter | [4, 8, 12, 16, 24, 32, 48] px | 700 |
| Animal recognition | Identify a single small animal silhouette | [4, 8, 12, 16, 24, 32, 48] px | 700 |
| Block recognition | Detect a single square block | [4, 8, 12, 16, 24, 32, 48] px | 700 |
| Colored block recognition | Identify block + its color | [4, 8, 12, 16, 24, 32, 48] px | 700 |
| Shape recognition | Identify a single geometric shape | [4, 8, 12, 16, 24, 32, 48] px | 700 |

**Total: 3 500 images**, 100 samples per (task, pixel size) pair, canvas 448 × 448.

## 1. Import Required Libraries

In [2]:
import sys
import json
import random
import math
from pathlib import Path
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

# Locate the repo root via the installed package location
import finesightbench as _fb
repo_root = Path(_fb.__file__).resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from finesightbench.perception.generator import generate_perception_dataset
from finesightbench.core.objects import (
    LETTERS, ANIMAL_TYPES, SHAPE_TYPES, TARGET_SIZES,
)
from finesightbench.core.colors import TARGET_COLORS

print("Package root :", repo_root)
print("Letters      :", LETTERS)
print("Animals      :", ANIMAL_TYPES)
print("Shapes       :", SHAPE_TYPES)
print("Colors       :", TARGET_COLORS)

Package root : /home/snt/projects_lujun/FineSightBench
Letters      : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Animals      : ['cat', 'dog', 'fish', 'bird', 'rabbit', 'turtle']
Shapes       : ['circle', 'triangle', 'square', 'star', 'diamond', 'pentagon', 'hexagon', 'cross']
Colors       : ['red', 'green', 'blue', 'yellow', 'cyan', 'magenta', 'orange', 'purple', 'pink', 'brown', 'black', 'gray']


## 2. Canvas and Global Configuration

In [3]:
# ── Dataset generation parameters ────────────────────────────────────────────
CANVAS_SIZE      = 448                          # width = height = 448 px
PIXEL_SIZES      = [4, 8, 12, 16, 24, 32, 48]  # 7 difficulty levels
NUM_PER_CONFIG   = 100                          # samples per (task, size) pair
SEED             = 42

# Expected counts
TASK_NAMES = ["letter", "animal", "block", "color_block", "shape"]
IMAGES_PER_TASK  = len(PIXEL_SIZES) * NUM_PER_CONFIG   # 700
TOTAL_IMAGES     = len(TASK_NAMES) * IMAGES_PER_TASK   # 3500

# Output directory  (relative to repo root)
OUTPUT_DIR = repo_root / "data" / "full_perception"

print(f"Canvas           : {CANVAS_SIZE} × {CANVAS_SIZE} px")
print(f"Pixel sizes      : {PIXEL_SIZES}")
print(f"Samples/config   : {NUM_PER_CONFIG}")
print(f"Images per task  : {IMAGES_PER_TASK}")
print(f"Tasks            : {TASK_NAMES}")
print(f"Total images     : {TOTAL_IMAGES}")
print(f"Output directory : {OUTPUT_DIR}")

Canvas           : 448 × 448 px
Pixel sizes      : [4, 8, 12, 16, 24, 32, 48]
Samples/config   : 100
Images per task  : 700
Tasks            : ['letter', 'animal', 'block', 'color_block', 'shape']
Total images     : 3500
Output directory : /home/snt/projects_lujun/FineSightBench/data/full_perception


## 3. Generate the Full Perception Dataset

The existing `generate_perception_dataset` function already supports all 5 task types and the `"difficulty"` mode that iterates over pixel sizes.  
We call it once with `num_per_config=100` to produce all 3 500 images.

> **Estimated time:** ~3–5 minutes (image I/O + SVG rasterization for animals).

In [4]:
import time

t0 = time.time()

labels_path = generate_perception_dataset(
    output_dir=OUTPUT_DIR,
    canvas_size=CANVAS_SIZE,
    sizes=PIXEL_SIZES,
    num_per_config=NUM_PER_CONFIG,
    seed=SEED,
    mode="difficulty",
)

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")
print(f"Labels saved to: {labels_path}")

[Perception] Generated 3500 samples (mode=difficulty) -> /home/snt/projects_lujun/FineSightBench/data/full_perception

Done in 14.0s
Labels saved to: /home/snt/projects_lujun/FineSightBench/data/full_perception/labels.json


## 4. Dataset Statistics

In [5]:
with open(labels_path) as f:
    dataset = json.load(f)

samples = dataset["samples"]
info    = dataset["dataset_info"]

print("=== Dataset Info ===")
print(f"  Name            : {info['name']}")
print(f"  Total samples   : {info['num_samples']}")
print(f"  Canvas size     : {info.get('target_sizes', PIXEL_SIZES)}")
print(f"  Generation mode : {info['generation_mode']}")
print()

# Count by task type
by_task: dict[str, int] = defaultdict(int)
by_size: dict[int, int] = defaultdict(int)
by_task_size: dict[tuple, int] = defaultdict(int)
by_difficulty: dict[str, int] = defaultdict(int)

for s in samples:
    task = s["task_type"]
    size = s["metadata"]["targets"][0]["size"]
    diff = s["difficulty"]
    by_task[task] += 1
    by_size[size]  += 1
    by_task_size[(task, size)] += 1
    by_difficulty[diff] += 1

print("=== Samples per Task ===")
for task, cnt in sorted(by_task.items()):
    print(f"  {task:<30s}: {cnt:>5d}")

print()
print("=== Samples per Pixel Size ===")
for size, cnt in sorted(by_size.items()):
    print(f"  {size:>3d} px : {cnt:>5d}")

print()
print("=== Difficulty Distribution ===")
for diff, cnt in sorted(by_difficulty.items()):
    print(f"  {diff:<10s}: {cnt:>5d}")

=== Dataset Info ===
  Name            : FineSight-Perception
  Total samples   : 3500
  Canvas size     : [4, 8, 12, 16, 24, 32, 48]
  Generation mode : difficulty

=== Samples per Task ===
  animal_recognition            :   700
  block_recognition             :   700
  color_block_recognition       :   700
  letter_recognition            :   700
  shape_recognition             :   700

=== Samples per Pixel Size ===
    4 px :   500
    8 px :   500
   12 px :   500
   16 px :   500
   24 px :   500
   32 px :   500
   48 px :   500

=== Difficulty Distribution ===
  easy      :  1000
  extreme   :   500
  hard      :  1000
  medium    :  1000


## 5. Visualize Sample Images

Show a 5 × 7 grid: one representative sample per (task, pixel size) combination.

In [6]:
matplotlib.use("Agg")   # keep headless; inline display handled by Jupyter

TASK_ORDER = [
    "letter_recognition",
    "animal_recognition",
    "block_recognition",
    "color_block_recognition",
    "shape_recognition",
]

# One representative sample per (task, size)
pool: dict[tuple, dict] = {}
for s in samples:
    task = s["task_type"]
    size = s["metadata"]["targets"][0]["size"]
    key  = (task, size)
    if key not in pool:
        pool[key] = s

rows = len(TASK_ORDER)
cols = len(PIXEL_SIZES)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.5))
fig.suptitle("FineSight-Perception — one sample per (task, pixel size)", fontsize=14, y=1.01)

for r, task in enumerate(TASK_ORDER):
    for c, size in enumerate(PIXEL_SIZES):
        ax = axes[r][c]
        key = (task, size)
        if key in pool:
            s = pool[key]
            img = Image.open(OUTPUT_DIR / s["image_path"])
            ax.imshow(img)
            ax.set_title(f"{size}px", fontsize=8)
        else:
            ax.axis("off")
        if c == 0:
            ax.set_ylabel(task.replace("_recognition", ""), fontsize=8, rotation=0,
                          labelpad=90, ha="left", va="center")
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
vis_path = OUTPUT_DIR / "preview_grid.png"
plt.savefig(vis_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Saved preview grid → {vis_path}")

Saved preview grid → /home/snt/projects_lujun/FineSightBench/data/full_perception/preview_grid.png


## 6. Inspect Sample Entries

Print a few representative Q&A pairs from each task type to confirm the question prompts and reference answers are correct.

In [7]:
seen_tasks: set[str] = set()
print(f"{'Task':<30s}  {'Size':>4}  {'Q (truncated)':<55s}  Answer")
print("-" * 110)
for s in samples:
    task = s["task_type"]
    size = s["metadata"]["targets"][0]["size"]
    # Show one example per task at 8 px and 32 px
    if (task, size) in [(t, sz) for t in TASK_ORDER for sz in (8, 32)] and \
       (task, size) not in seen_tasks:
        seen_tasks.add((task, size))
        q_short = s["question"][:55].replace("\n", " ")
        print(f"{task:<30s}  {size:>4}px  {q_short:<55s}  {s['answer']}")

Task                            Size  Q (truncated)                                            Answer
--------------------------------------------------------------------------------------------------------------
letter_recognition                 8px  What letter is displayed in the image? Answer with a si  M
letter_recognition                32px  What letter is displayed in the image? Answer with a si  R
animal_recognition                 8px  What animal is shown in the image? Answer with one of:   bird
animal_recognition                32px  What animal is shown in the image? Answer with one of:   turtle
block_recognition                  8px  Is there a square block in the image? Describe what you  yes
block_recognition                 32px  Is there a square block in the image? Describe what you  yes
color_block_recognition            8px  What color is the block in the image? Answer with one o  green
color_block_recognition           32px  What color is the block in the image? 

## 7. Export Metadata to JSON and CSV

The primary `labels.json` is already written by the generator.  
Here we also export a flat `metadata.csv` with one row per sample for easy inspection in spreadsheet tools.

In [8]:
import csv

csv_path = OUTPUT_DIR / "metadata.csv"
fieldnames = [
    "image_id", "image_path", "task_type", "pixel_size",
    "difficulty", "question", "answer",
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for s in samples:
        size = s["metadata"]["targets"][0]["size"]
        writer.writerow({
            "image_id"  : s["image_id"],
            "image_path": s["image_path"],
            "task_type" : s["task_type"],
            "pixel_size": size,
            "difficulty": s["difficulty"],
            "question"  : s["question"],
            "answer"    : s["answer"],
        })

print(f"CSV written : {csv_path}  ({len(samples)} rows)")
print(f"JSON written: {labels_path}")

# Summary table
print()
print(f"{'Task':<30s}", end="")
for sz in PIXEL_SIZES:
    print(f"  {sz:>3}px", end="")
print("   Total")
print("-" * (30 + len(PIXEL_SIZES) * 6 + 8))
for task in TASK_ORDER:
    print(f"{task:<30s}", end="")
    row_total = 0
    for sz in PIXEL_SIZES:
        cnt = by_task_size.get((task, sz), 0)
        print(f"  {cnt:>4d}", end="")
        row_total += cnt
    print(f"  {row_total:>6d}")
print("-" * (30 + len(PIXEL_SIZES) * 6 + 8))
print(f"{'Total':<30s}", end="")
col_totals = [sum(by_task_size.get((t, sz), 0) for t in TASK_ORDER) for sz in PIXEL_SIZES]
for ct in col_totals:
    print(f"  {ct:>4d}", end="")
print(f"  {sum(col_totals):>6d}")

CSV written : /home/snt/projects_lujun/FineSightBench/data/full_perception/metadata.csv  (3500 rows)
JSON written: /home/snt/projects_lujun/FineSightBench/data/full_perception/labels.json

Task                              4px    8px   12px   16px   24px   32px   48px   Total
--------------------------------------------------------------------------------
letter_recognition               100   100   100   100   100   100   100     700
animal_recognition               100   100   100   100   100   100   100     700
block_recognition                100   100   100   100   100   100   100     700
color_block_recognition          100   100   100   100   100   100   100     700
shape_recognition                100   100   100   100   100   100   100     700
--------------------------------------------------------------------------------
Total                            500   500   500   500   500   500   500    3500


## 8. Re-label: JSON-format Q&A + JSONL Export

For each sample:
- **Question** is updated to instruct the VLM to reply with a strict JSON object.
- **Answer** becomes a key-value `dict` (instead of a plain string) so that VLM output can be parsed and compared field-by-field.

Output file: `labels.jsonl` — one JSON object per line, fully self-contained.

| Task | JSON answer schema |
|------|--------------------|
| `letter_recognition` | `{"letter": "M"}` |
| `animal_recognition` | `{"animal": "cat"}` |
| `block_recognition` | `{"present": "yes"}` |
| `color_block_recognition` | `{"color": "red"}` |
| `shape_recognition` | `{"shape": "circle"}` |

In [9]:
import json as _json
import finesightbench as _fb
from pathlib import Path
from finesightbench.core.objects import LETTERS, ANIMAL_TYPES, SHAPE_TYPES
from finesightbench.core.colors import TARGET_COLORS

# ── vocab strings ─────────────────────────────────────────────────────────────
_LETTER_CHOICES = ", ".join(LETTERS)
_ANIMAL_CHOICES = ", ".join(ANIMAL_TYPES)
_SHAPE_CHOICES  = ", ".join(SHAPE_TYPES)
_COLOR_CHOICES  = ", ".join(c for c in TARGET_COLORS if c != "black")

_PERC_OUTPUT_DIR = Path(_fb.__file__).resolve().parent.parent / "data" / "full_perception"


def _perception_qa(s: dict) -> tuple[str, dict]:
    """Return (new_question, answer_dict) for a perception sample."""
    task = s["task_type"]

    if task == "letter_recognition":
        q = (
            "What letter is displayed in the image? "
            'Answer in JSON format only: {"letter": "<X>"} '
            f"where <X> is one uppercase letter from: {_LETTER_CHOICES}."
        )
        a = {"letter": s["answer"]}

    elif task == "animal_recognition":
        q = (
            "What animal is shown in the image? "
            'Answer in JSON format only: {"animal": "<type>"} '
            f"where <type> is one of: {_ANIMAL_CHOICES}."
        )
        a = {"animal": s["answer"]}

    elif task == "block_recognition":
        q = (
            "Is there a square block visible in the image? "
            'Answer in JSON format only: {"present": "yes"} or {"present": "no"}.'
        )
        a = {"present": s["answer"]}          # original answer is always "yes"

    elif task == "color_block_recognition":
        q = (
            "What color is the block in the image? "
            'Answer in JSON format only: {"color": "<name>"} '
            f"where <name> is one of: {_COLOR_CHOICES}."
        )
        a = {"color": s["answer"]}

    elif task == "shape_recognition":
        q = (
            "What geometric shape is displayed in the image? "
            'Answer in JSON format only: {"shape": "<type>"} '
            f"where <type> is one of: {_SHAPE_CHOICES}."
        )
        a = {"shape": s["answer"]}

    else:
        q = s["question"]
        a = {"value": s["answer"]}

    return q, a


# ── load existing labels ──────────────────────────────────────────────────────
with open(_PERC_OUTPUT_DIR / "labels.json", encoding="utf-8") as _f:
    _data = _json.load(_f)
_samples = _data["samples"]

# ── write JSONL ───────────────────────────────────────────────────────────────
jsonl_path = _PERC_OUTPUT_DIR / "labels.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as _f:
    for s in _samples:
        new_q, new_a = _perception_qa(s)
        row = dict(s)
        row["question"]      = new_q
        row["answer"]        = new_a
        row["answer_format"] = "json"
        _f.write(_json.dumps(row, ensure_ascii=False) + "\n")

print(f"Written {len(_samples)} lines → {jsonl_path}")

# ── spot-check one sample per task ───────────────────────────────────────────
_seen: set[str] = set()
for s in _samples:
    task = s["task_type"]
    if task not in _seen:
        _seen.add(task)
        new_q, new_a = _perception_qa(s)
        print(f"\n[{task}]")
        print(f"  Q : {new_q[:110]}")
        print(f"  A : {new_a}")

Written 3500 lines → /home/snt/projects_lujun/FineSightBench/data/full_perception/labels.jsonl

[letter_recognition]
  Q : What letter is displayed in the image? Answer in JSON format only: {"letter": "<X>"} where <X> is one uppercas
  A : {'letter': 'A'}

[animal_recognition]
  Q : What animal is shown in the image? Answer in JSON format only: {"animal": "<type>"} where <type> is one of: ca
  A : {'animal': 'bird'}

[block_recognition]
  Q : Is there a square block visible in the image? Answer in JSON format only: {"present": "yes"} or {"present": "n
  A : {'present': 'yes'}

[color_block_recognition]
  Q : What color is the block in the image? Answer in JSON format only: {"color": "<name>"} where <name> is one of: 
  A : {'color': 'gray'}

[shape_recognition]
  Q : What geometric shape is displayed in the image? Answer in JSON format only: {"shape": "<type>"} where <type> i
  A : {'shape': 'circle'}
